In [1]:
import torch
import torch.nn as nn
import torch.nn as nn
import numpy as np

##훈련 데이터 전처리

In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print('문자 집합의 크기:{}'.format(vocab_size))

문자 집합의 크기:5


하이퍼파라미터를 정의한다. 이때 입력을 원-핫 벡터를 사용할 것으로 입력의 크기는 문자 집합의 크기여야 한다.

In [3]:
input_size = vocab_size
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [6]:
char_to_index = dict((c,i) for i,c in enumerate(char_vocab))
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


나중에 예측 결과를 다시 문자 시퀀스로 보기 위해 반대로 정수로부터 문자를 얻을 수 있는 index_to_char을 만듦

In [7]:
index_to_char={}
for key, value in char_to_index.items():
  index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [9]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


파이토치의 nn.RNN()은 기본적으로 3차원 텐서를 입력받는다. 그렇기 때문에 배치 차원을 추가해준다.

In [10]:
#배치 차원 추가
#텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있음
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [11]:
#입력 시퀀스의 각 문자들을 원-핫 벡터로 바꿔줌
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [12]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipykernel_1013/2348034151.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  X = torch.FloatTensor(x_one_hot)


In [13]:
print('훈련 데이터의 크기: {}'.format(X.shape))
print('레이블의 크기:{}'.format(Y.shape))

훈련 데이터의 크기: torch.Size([1, 5, 5])
레이블의 크기:torch.Size([1, 5])


##모델 구현

fc는 완전 연결층을 의미하며 출력층으로 사용

In [15]:
class Net(torch.nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(Net, self).__init__()
    self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True)
#RNN 셀 구현
    self.fc = torch.nn.Linear(hidden_size, output_size, bias=True)
#출력층 구현
  def forward(self, x):
    x, _status = self.rnn(x)
    x = self.fc(x)
    return x

In [17]:
net = Net(input_size, hidden_size, output_size)

In [18]:
outputs = net(X)
print(outputs.shape)

torch.Size([1, 5, 5])


In [19]:
torch.Size([1,5,5])

torch.Size([1, 5, 5])

(1,5,5)의 크기를 가지는데 각각 배치 차원, 시점, 출력의 크기이다. 나중에 정확도를 측정할 때는 이를 모두 펼쳐서 계산하는데 이때는 view를 사용하여 배치 차원과 시점 차원을 하나로 만듦

In [20]:
print(outputs.view(-1, input_size).shape)

torch.Size([5, 5])


In [21]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


레이블 데이터는 (1,5)의 크기를 가지는데 마찬가지로 나중에 정확도를 측정할 때는 이걸 펼쳐서 계산할 예정.

이 경우 (5)의 크기를 가지게 됨.

이제 옵티마이저와 손실 함수를 정의한다.

In [23]:
import torch.optim as optim

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [26]:
for i in range(100):
  optimizer.zero_grad()
  outputs = net(X)
  loss = criterion(outputs.view(-1, input_size), Y.view(-1))
  #view(-1)하는 이유는 Batch 차원 제거를 위해
  loss.backward()  #기울기 계산
  optimizer.step()   #아까 optimizer선언 시 넣어둔 파라미터 업데이트

  #아래 세 줄을 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드
  result = outputs.data.numpy().argmax(axis=2)
  result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
  print(i, 'loss:', loss.item(), 'prediction:', result, 'true Y:', y_data,'prediction str:', result_str)

0 loss: 1.6719224452972412 prediction: [[3 3 3 3 3]] true Y: [[4, 4, 3, 2, 0]] prediction str: lllll
1 loss: 1.3586366176605225 prediction: [[4 4 4 2 3]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppel
2 loss: 1.177765965461731 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
3 loss: 1.0044318437576294 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
4 loss: 0.842037558555603 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
5 loss: 0.6973100304603577 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
6 loss: 0.5547345280647278 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
7 loss: 0.43472662568092346 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
8 loss: 0.34010934829711914 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
9 loss: 0.2583421468734741 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction s

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim

In [30]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")


In [31]:
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩


In [32]:
print(char_dic) # 공백도 여기서는 하나의 원소

{'u': 0, 't': 1, 'y': 2, 'i': 3, '.': 4, 'd': 5, 's': 6, 'b': 7, 'a': 8, 'k': 9, 'e': 10, 'h': 11, 'p': 12, 'm': 13, 'c': 14, 'o': 15, 'n': 16, 'g': 17, ',': 18, 'f': 19, 'l': 20, "'": 21, ' ': 22, 'r': 23, 'w': 24}


In [33]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


문자 집합의 크기는 25이고, 입력을 원-핫 벡터로 사용할 것이므로 이는 매 시점마다 들어갈 입력의 크기이기도 하다.

이제 하이퍼파라미터를 설정한다.

hidden_size를 입력의 크기와 동일하게 줬는데, 이는 사용자의 선택으로 다른 값을 줘도 무방하다.

sequence_length라는 변수는 우리가 앞서 만든 샘플을 10개 단위로 끊어서 샘플을 만들 예정이기 때문

In [35]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1

In [36]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [37]:
print(x_data[0])
print(y_data[0])

[3, 19, 22, 2, 15, 0, 22, 24, 8, 16]
[19, 22, 2, 15, 0, 22, 24, 8, 16, 1]


In [38]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data]
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [39]:
print('훈련 데이터의 크기:{}'.format(X.shape))
print('레이블의 크기:{}'.format(Y.shape))

훈련 데이터의 크기:torch.Size([170, 10, 25])
레이블의 크기:torch.Size([170, 10])


원-핫 인코딩 된 결과를 보기 위해서 첫번째 샘플만 출력

In [40]:
print(X[0])

tensor([[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,

## 모델 구현

In [41]:
class Net(torch.nn.Module):
  def __init__(self, input_dim, hidden_dim, layers):
    super(Net, self).__init__()
    self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
    self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

  def forward(self, x):
    x,_status = self.rnn(x)
    x = self.fc(x)
    return x

In [42]:
net = Net(dic_size, hidden_size,2)

In [43]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [44]:
outputs = net(X)
print(outputs.shape)

torch.Size([170, 10, 25])


In [45]:
print(outputs.view(-1, dic_size).shape)

torch.Size([1700, 25])


In [47]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [52]:
for i in range(100):
  optimizer.zero_grad()
  outputs = net(X)
  loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
  loss.backward()
  optimizer.step()

  #results의 텐서 크기는 (170,10)
  results = outputs.argmax(dim=2)
  predict_str=""
  for j, result in enumerate(results):
    if j == 0: # 처음에는 예측 결과를 전부 가져오지만
          predict_str += ''.join([char_set[t] for t in result])
    else: # 그 다음에는 마지막 글자만 반복 추가
          predict_str += char_set[result[-1]]

print(predict_str)

f to  want to build a ship, don't arum up people together te collect word and don't assign them tasks and dork, but rather teach them ta long for thersndless immensity of themseac
